# Tabular Summary Example
## 1. Data Profiler with Python
## 2. Pass JSON Profile to LLM

In [1]:
from openai import AzureOpenAI
from dotenv import load_dotenv
import os
import pandas as pd
import tiktoken

load_dotenv()

True

In [2]:
azure_endpoint = os.getenv("AZURE_ENDPOINT")
api_key = os.getenv("API_KEY")
api_version = os.getenv("API_VERSION")
deployment_name = os.getenv("DEPLOYMENT_NAME")

In [3]:
client = AzureOpenAI(
    azure_endpoint = azure_endpoint,
    api_key = api_key,
    api_version = api_version
)

# Data Profiler

In [4]:
import re
import math
import hashlib
import json
import numpy as np
import pandas as pd

# =========================
# ===== Helper Utils ======
# =========================

def _hashish(x):
    if pd.isna(x): return None
    return hashlib.sha1(str(x).encode()).hexdigest()[:10]

def _entropy_from_counts(counts):
    counts = counts.astype(float)
    s = counts.sum()
    if s <= 0: return 0.0
    p = counts / s
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def _safe_quantiles(x, qs):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {f"p{int(q*100):02d}": None for q in qs}
    vals = np.quantile(x.to_numpy(), qs)
    return {f"p{int(q*100):02d}": float(v) for q, v in zip(qs, vals)}

def _hist_by_quantiles(x, nbins):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    x = x[np.isfinite(x)]
    if len(x) == 0 or nbins < 1:
        return {"bins": [], "counts": []}
    edges = np.unique(np.quantile(x, np.linspace(0, 1, nbins + 1)))
    if len(edges) < 2:
        return {"bins": [float(x.iloc[0]), float(x.iloc[0])], "counts": [int(len(x))]}
    counts, edges = np.histogram(x, bins=edges)
    return {"bins": [float(b) for b in edges], "counts": [int(c) for c in counts]}

def _skew_kurt(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 3:
        return 0.0, 0.0
    m = np.mean(x)
    s = np.std(x, ddof=1)
    if s == 0:
        return 0.0, 0.0
    z = (x - m) / s
    skew = float((z**3).mean())
    kurt = float((z**4).mean() - 3.0)  # excess kurtosis
    return skew, kurt

def _spearman_abs(a, b):
    sa = pd.Series(a).rank(method="average", na_option="keep")
    sb = pd.Series(b).rank(method="average", na_option="keep")
    return float(abs(sa.corr(sb)))

def _cramers_v_bias_corrected(ct: np.ndarray):
    n = ct.sum()
    if n == 0: return 0.0
    row_sum = ct.sum(axis=1)
    col_sum = ct.sum(axis=0)
    expected = np.outer(row_sum, col_sum) / n
    with np.errstate(divide='ignore', invalid='ignore'):
        chi2 = np.nansum((ct - expected) ** 2 / expected)
    phi2 = max(chi2 / n, 0.0)
    r, k = ct.shape
    phi2corr = max(0.0, phi2 - ((k-1)*(r-1))/(n-1)) if n > 1 else 0.0
    rcorr = r - ((r-1)**2)/(n-1) if n > 1 else r
    kcorr = k - ((k-1)**2)/(n-1) if n > 1 else k
    denom = max(min(rcorr-1, kcorr-1), 1e-12)
    return float(np.sqrt(phi2corr / denom))

def _correlation_ratio(categories, values):
    c = pd.Series(categories)
    v = pd.to_numeric(pd.Series(values), errors="coerce")
    mask = (~c.isna()) & np.isfinite(v)
    c, v = c[mask], v[mask]
    if len(v) < 2: return 0.0
    groups = v.groupby(c)
    n = float(len(v))
    mu = float(v.mean())
    ssb = float((groups.count() * (groups.mean() - mu) ** 2).sum())
    sst = float(((v - mu) ** 2).sum())
    return float(ssb / sst) if sst > 0 else 0.0

def _is_texty_series(s, min_avg_len=15, min_alpha_share=0.5):
    if s.dtype == object or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
        ss = s.dropna().astype(str)
        if len(ss) == 0: return False
        lens = ss.str.len()
        avg = lens.mean()
        # alpha share: fraction of alphabetic chars
        sample = ''.join(ss.sample(min(200, len(ss)), random_state=0).tolist())
        alpha = sum(ch.isalpha() for ch in sample)
        share = alpha / max(1, len(sample))
        return (avg >= min_avg_len) and (share >= min_alpha_share)
    return False

def _token_topk(ss, k=15):
    tokens = re.split(r"\W+", " ".join(ss).lower())
    tokens = [t for t in tokens if len(t) >= 3]
    if not tokens:
        return []
    vc = pd.Series(tokens).value_counts().head(k)
    tot = int(vc.sum())
    return [{"token": t, "count": int(c), "share": float(c/tot)} for t, c in vc.items()]

def _contingency_table(si, sj, *, max_dim=20, max_cells=250, normalize=None):
    """
    Build a compact contingency table for si x sj if within size caps.
    normalize: None | 'row' | 'col' | 'all'
    """
    ct = pd.crosstab(si, sj, dropna=True)
    r, c = ct.shape
    if r == 0 or c == 0: 
        return None
    if r > max_dim or c > max_dim or (r * c) > max_cells:
        return None
    if normalize == "row":
        tbl = ct.div(ct.sum(axis=1), axis=0).fillna(0.0)
        fmt = lambda v: float(v)
    elif normalize == "col":
        tbl = ct.div(ct.sum(axis=0), axis=1).fillna(0.0)
        fmt = lambda v: float(v)
    elif normalize == "all":
        total = ct.values.sum()
        tbl = ct / (total if total else 1)
        fmt = lambda v: float(v)
    else:
        tbl = ct
        fmt = lambda v: int(v)
    return {str(rk): {str(ck): fmt(v) for ck, v in row.items()} for rk, row in tbl.iterrows()}

# ==============================
# ===== The Summarizer API =====
# ==============================

def summarize_df(
    df: pd.DataFrame,
    *,
    target: str | None = None,           # optional target column
    mask_id_like: bool = True,
    sample_rows_for_examples: int = 5000,
    max_categories: int = 15,
    nbins_numeric: int = 12,
    interaction_topk: int = 30,
    min_abs_r: float = 0.5,
    min_cramers_v: float = 0.2,
    min_eta: float = 0.1,
    max_cat_card_for_pairs: int = 80,    # skip huge categoricals in pairwise stats
    max_missing_pairs: int = 20,         # top-k missingness co-occurrence pairs
    text_token_topk: int = 15,
    include_contingencies: bool = True,  # 🔸 NEW: include contingency tables
    contingency_max_dim: int = 20,       # 🔸 NEW: cap rows/cols
    contingency_max_cells: int = 250,    # 🔸 NEW: cap total cells
    contingency_normalize: str | None = None,  # 🔸 NEW: None | 'row' | 'col' | 'all'
    random_state: int = 0
):
    """
    Create a rich, LLM-friendly JSON summary of a pandas DataFrame.

    Returns a JSON-serializable dict with:
      - table_card
      - columns: per-column stats
      - interactions: numeric-numeric (pearson/spearman), categorical-categorical (Cramér's V [+ contingency table]), categorical-numeric (η)
      - missingness: per-column, row-level, co-null pairs
      - text_profiles
      - time_card
      - target_card (optional)
    """
    rng = np.random.default_rng(random_state)
    df = df.copy()

    # -------- Privacy: mask ID-like columns --------
    id_like = {c for c in df.columns if any(k in c.lower() for k in ("id", "uuid", "guid"))}
    if mask_id_like:
        for c in id_like:
            df[c] = df[c].map(_hashish)

    # -------- Table-level quality --------
    n_rows, n_cols = df.shape
    dup_row_frac = float(df.duplicated().mean()) if n_rows else 0.0
    key_dups = {}
    for c in id_like:
        notna = df[c].notna()
        if notna.any():
            key_dups[c] = float(df.loc[notna, c].duplicated().mean())
    time_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
    time_col = time_cols[0] if time_cols else None

    table_card = dict(
        n_rows=int(n_rows),
        n_cols=int(n_cols),
        duplicate_row_frac=dup_row_frac,
        id_like=list(sorted(id_like)),
        duplicate_key_frac=key_dups,
        has_time_column=bool(time_col),
        time_column=time_col
    )

    # -------- Example rows (small sample) --------
    if n_rows > 0:
        ex_idx = rng.choice(df.index, size=min(sample_rows_for_examples, n_rows), replace=False)
        examples_df = df.loc[ex_idx].head(5)
        example_rows = examples_df.astype(object).astype(str).to_dict(orient="records")
    else:
        example_rows = []

    # -------- Per-column cards --------
    columns = []
    quantiles = (0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99)

    for c in df.columns:
        s = df[c]
        card = {"name": c, "null_frac": float(s.isna().mean())}
        card["distinct"] = int(s.nunique(dropna=True))
        card["constant"] = bool(card["distinct"] <= 1)

        if pd.api.types.is_numeric_dtype(s):
            card["role"] = "numeric"
            sn = pd.to_numeric(s, errors="coerce")
            finite = sn[np.isfinite(sn)]
            card["count"] = int(sn.notna().sum())
            if len(finite) > 0:
                card["range"] = [float(finite.min()), float(finite.max())]
                card["quantiles"] = _safe_quantiles(finite, quantiles)
                card["mean"] = float(finite.mean())
                card["std"] = float(finite.std(ddof=1)) if len(finite) > 1 else 0.0
                sk, ku = _skew_kurt(finite)
                card["skew"], card["kurtosis_excess"] = float(sk), float(ku)
                q95 = card["quantiles"]["p95"]; q05 = card["quantiles"]["p05"]
                if q95 is not None and q05 is not None:
                    card["tail_share_gt_p95"] = float((finite > q95).mean())
                    card["tail_share_lt_p05"] = float((finite < q05).mean())
                card["zero_share"] = float((finite == 0).mean())
                card["negative_share"] = float((finite < 0).mean())
                p25 = card["quantiles"]["p25"]; p75 = card["quantiles"]["p75"]
                if p25 is not None and p75 is not None:
                    iqr = p75 - p25
                    lf, uf = p25 - 1.5*iqr, p75 + 1.5*iqr
                    card["outliers_tukey"] = {"lower": int((finite < lf).sum()),
                                              "upper": int((finite > uf).sum())}
                card["histogram"] = _hist_by_quantiles(finite, nbins_numeric)
            else:
                card["range"] = [None, None]
                card["quantiles"] = _safe_quantiles([], quantiles)
                card["mean"] = card["std"] = 0.0
                card["histogram"] = {"bins": [], "counts": []}

        elif pd.api.types.is_bool_dtype(s):
            card["role"] = "boolean"
            vc = s.value_counts(dropna=True)
            card["counts"] = {str(k): int(v) for k, v in vc.items()}

        elif pd.api.types.is_datetime64_any_dtype(s):
            card["role"] = "datetime"
            notna = s.dropna()
            if len(notna) > 0:
                card["range"] = [notna.min().isoformat(), notna.max().isoformat()]
                diffs = notna.sort_values().diff().dropna().dt.total_seconds()
                if len(diffs) > 0:
                    med = np.median(diffs)
                    if med < 60: grain = "second"
                    elif med < 3600: grain = "minute"
                    elif med < 86400: grain = "hour"
                    elif med < 604800: grain = "day"
                    elif med < 2678400: grain = "week"
                    else: grain = "month+"
                    card["inferred_grain"] = grain
                idx = notna.dt
                wd_share = float((idx.dayofweek < 5).mean())
                card["weekday_share"] = wd_share
                by_day = notna.dt.floor("D").value_counts()
                by_week = notna.dt.to_period("W").value_counts()
                card["daily_counts_mean_std"] = [
                    float(by_day.mean()) if len(by_day) else 0.0,
                    float(by_day.std(ddof=1)) if len(by_day) > 1 else 0.0
                ]
                card["weekly_counts_mean_std"] = [
                    float(by_week.mean()) if len(by_week) else 0.0,
                    float(by_week.std(ddof=1)) if len(by_week) > 1 else 0.0
                ]
            else:
                card["range"] = [None, None]

        else:
            if _is_texty_series(s):
                card["role"] = "text"
                ss = s.dropna().astype(str)
                lens = ss.str.len()
                card["length_mean_std"] = [float(lens.mean()), float(lens.std(ddof=1)) if len(lens) > 1 else 0.0]
                sample = ''.join(ss.sample(min(200, len(ss)), random_state=0).tolist())
                total = max(1, len(sample))
                card["char_shares"] = {
                    "alpha": sum(ch.isalpha() for ch in sample)/total,
                    "digit": sum(ch.isdigit() for ch in sample)/total,
                    "space": sum(ch.isspace() for ch in sample)/total
                }
                card["token_topk"] = _token_topk(ss.sample(min(1000, len(ss)), random_state=0).tolist())
            else:
                card["role"] = "categorical"
                ss = s.dropna().astype(str)
                vc = ss.value_counts()
                card["cardinality"] = int(vc.shape[0])
                top = vc.head(max_categories)
                total = int(vc.sum()) if len(vc) else 0
                card["top_k"] = [{"value": k, "count": int(v), "share": float(v/total) if total else 0.0}
                                 for k, v in top.items()]
                tail = int(vc.iloc[max_categories:].sum()) if vc.shape[0] > max_categories else 0
                card["tail_other_count"] = tail if tail else 0
                card["entropy_bits"] = _entropy_from_counts(vc.values) if len(vc) else 0.0

        columns.append(card)

    # -------- Missingness patterns --------
    col_null = {c: float(df[c].isna().mean()) for c in df.columns}
    row_any_null_frac = float(df.isna().any(axis=1).mean()) if n_rows else 0.0
    row_all_null_frac = float(df.isna().all(axis=1).mean()) if n_rows else 0.0
    miss_pairs = []
    if n_cols >= 2 and n_rows:
        null_mat = df.isna()
        cols = list(df.columns)
        if n_rows > 200_000:
            ridx = rng.choice(df.index, size=200_000, replace=False)
            null_mat = null_mat.loc[ridx]
        for i in range(len(cols)):
            a = null_mat[cols[i]].to_numpy()
            ai = a.mean()
            for j in range(i+1, len(cols)):
                b = null_mat[cols[j]].to_numpy()
                co = float((a & b).mean())
                lift = co / max(ai * b.mean(), 1e-12)
                if co > 0:
                    miss_pairs.append({"pair": [cols[i], cols[j]], "co_null_frac": co, "lift": lift})
        miss_pairs.sort(key=lambda x: (-x["co_null_frac"], -x["lift"]))
        miss_pairs = miss_pairs[:max_missing_pairs]

    missingness = dict(
        per_column=col_null,
        row_any_null_frac=row_any_null_frac,
        row_all_null_frac=row_all_null_frac,
        top_conull_pairs=miss_pairs
    )

    # -------- Interactions --------
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols = [c for c in df.columns 
                if (pd.api.types.is_categorical_dtype(df[c]) 
                    or df[c].dtype == object or pd.api.types.is_bool_dtype(df[c]))
                and not pd.api.types.is_datetime64_any_dtype(df[c])]

    interactions = {
        "numeric_numeric": {"pearson_top": [], "spearman_top": []},
        "categorical_categorical": {"cramers_v_top": []},
        "categorical_numeric": {"eta_top": []}
    }

    # numeric-numeric
    if len(num_cols) >= 2:
        num_df = df[num_cols].apply(pd.to_numeric, errors="coerce")
        pear = num_df.corr(numeric_only=True).abs()
        pairs_p = []
        cols = pear.columns.tolist()
        for i in range(len(cols)):
            for j in range(i+1, len(cols)):
                r = pear.iloc[i, j]
                if pd.notna(r) and r >= min_abs_r:
                    pairs_p.append({"pair": [cols[i], cols[j]], "abs_r": float(r), "method": "pearson"})
        pairs_p.sort(key=lambda x: -x["abs_r"])
        interactions["numeric_numeric"]["pearson_top"] = pairs_p[:interaction_topk]

        pairs_s = []
        for i in range(len(cols)):
            a = num_df[cols[i]].to_numpy()
            for j in range(i+1, len(cols)):
                b = num_df[cols[j]].to_numpy()
                rho = _spearman_abs(a, b)
                if not math.isnan(rho) and rho >= min_abs_r:
                    pairs_s.append({"pair": [cols[i], cols[j]], "abs_r": float(rho), "method": "spearman"})
        pairs_s.sort(key=lambda x: -x["abs_r"])
        interactions["numeric_numeric"]["spearman_top"] = pairs_s[:interaction_topk]

    # categorical-categorical (+ contingency tables)
    if len(cat_cols) >= 2:
        cc = []
        usable = []
        for c in cat_cols:
            s = df[c]
            card = int(s.dropna().nunique())
            if 1 < card <= max_cat_card_for_pairs:
                usable.append(c)
        for i in range(len(usable)):
            si = df[usable[i]].astype("string")
            for j in range(i+1, len(usable)):
                sj = df[usable[j]].astype("string")
                ct_df = pd.crosstab(si, sj, dropna=True)
                if ct_df.size == 0:
                    continue
                v = _cramers_v_bias_corrected(ct_df.to_numpy())
                if not math.isnan(v) and v >= min_cramers_v:
                    entry = {"pair": [usable[i], usable[j]], "cramers_v": float(v)}
                    if include_contingencies:
                        table = _contingency_table(
                            si, sj,
                            max_dim=contingency_max_dim,
                            max_cells=contingency_max_cells,
                            normalize=contingency_normalize
                        )
                        if table is not None:
                            entry["contingency_table"] = table
                    cc.append(entry)
        cc.sort(key=lambda x: -x["cramers_v"])
        interactions["categorical_categorical"]["cramers_v_top"] = cc[:interaction_topk]

    # categorical-numeric (η)
    if len(cat_cols) >= 1 and len(num_cols) >= 1:
        cn = []
        for ccat in cat_cols:
            s_cat = df[ccat]
            card = int(s_cat.dropna().nunique())
            if card <= 1 or card > max_cat_card_for_pairs:
                continue
            for cnum in num_cols:
                s_num = pd.to_numeric(df[cnum], errors="coerce")
                eta = _correlation_ratio(s_cat, s_num)
                if not math.isnan(eta) and eta >= min_eta:
                    cn.append({"pair": [ccat, cnum], "eta": float(eta)})
        cn.sort(key=lambda x: -x["eta"])
        interactions["categorical_numeric"]["eta_top"] = cn[:interaction_topk]

    # -------- Text profiles (index by name) --------
    text_profiles = {
        c: next(card for card in columns if card["name"] == c)
        for c in [card["name"] for card in columns if card.get("role") == "text"]
    }

    # -------- Optional: target insights --------
    target_card = None
    if target is not None and target in df.columns:
        y = df[target]
        tc = {"name": target}
        if pd.api.types.is_numeric_dtype(y):
            tc["type"] = "regression"
            assoc = []
            for c in num_cols:
                if c == target: 
                    continue
                r = float(abs(pd.to_numeric(df[c], errors="coerce").corr(pd.to_numeric(y, errors="coerce"))))
                if not math.isnan(r):
                    assoc.append({"feature": c, "abs_pearson": r})
            assoc.sort(key=lambda x: -x["abs_pearson"])
            tc["numeric_correlations_top"] = assoc[:interaction_topk]
            cat_eta = []
            for c in cat_cols:
                eta = _correlation_ratio(df[c], pd.to_numeric(y, errors="coerce"))
                if not math.isnan(eta):
                    cat_eta.append({"feature": c, "eta": float(eta)})
            cat_eta.sort(key=lambda x: -x["eta"])
            tc["categorical_eta_top"] = cat_eta[:interaction_topk]
        else:
            tc["type"] = "classification"
            y_str = y.astype("string")
            cc2 = []
            for c in cat_cols:
                ct = pd.crosstab(df[c].astype("string"), y_str, dropna=True).to_numpy()
                if ct.size:
                    v = _cramers_v_bias_corrected(ct)
                    if not math.isnan(v):
                        cc2.append({"feature": c, "cramers_v": float(v)})
            cc2.sort(key=lambda x: -x["cramers_v"])
            tc["categorical_cramers_v_top"] = cc2[:interaction_topk]
            ne = []
            for c in num_cols:
                eta = _correlation_ratio(y_str, pd.to_numeric(df[c], errors="coerce"))
                if not math.isnan(eta):
                    ne.append({"feature": c, "eta": float(eta)})
            ne.sort(key=lambda x: -x["eta"])
            tc["numeric_eta_top"] = ne[:interaction_topk]
        target_card = tc

    # -------- Time summary (if any) --------
    time_card = None
    if time_col is not None:
        ts = df[time_col].dropna().sort_values()
        if len(ts) > 1:
            diffs = ts.diff().dropna().dt.total_seconds()
            gap_share = float((diffs > np.quantile(diffs, 0.95)).mean()) if len(diffs) else 0.0
            time_card = {
                "column": time_col,
                "min": ts.min().isoformat(),
                "max": ts.max().isoformat(),
                "median_step_seconds": float(np.median(diffs)) if len(diffs) else None,
                "extreme_gap_share": gap_share
            }

    # -------- Assemble --------
    out = {
        "table_card": table_card,
        "example_rows": example_rows,
        "columns": columns,
        "missingness": missingness,
        "interactions": interactions,
        "text_profiles": text_profiles,
        "time_card": time_card,
        "target_card": target_card
    }
    return out

In [5]:
def load_data_table(file_path: str, **read_kwargs) -> pd.DataFrame:
    """
    Load a tabular file into a pandas DataFrame.
    Supports CSV, TSV, Excel, Parquet, Feather, and JSON.
    Extra kwargs are passed to the pandas reader.
    """
    ext = os.path.splitext(file_path)[1].lower()

    if ext in (".csv", ".txt"):
        return pd.read_csv(file_path, **read_kwargs)
    elif ext == ".tsv":
        return pd.read_csv(file_path, sep="\t", **read_kwargs)
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(file_path, **read_kwargs)
    elif ext == ".parquet":
        return pd.read_parquet(file_path, **read_kwargs)
    elif ext == ".feather":
        return pd.read_feather(file_path, **read_kwargs)
    elif ext == ".json":
        return pd.read_json(file_path, **read_kwargs)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")
    

In [6]:
def get_token_count(text, model="gpt-4"):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(text)
    return len(tokens)

In [7]:
file_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/Stuff/hmeq.csv"
df = load_data_table(file_path)

In [8]:
tabular_summary = summarize_df(df)
tabular_summary_str = json.dumps(tabular_summary, indent=2, default=str)
get_token_count(tabular_summary_str)

/var/folders/21/snnzgwl511sfthgmb96x6bmw0000gn/T/ipykernel_18455/850342753.py:354: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if (pd.api.types.is_categorical_dtype(df[c])


7460

# LLM with JSON Data Profile

In [9]:
def llm_tabular_summary(tabular_summary_str):

    tabular_data_prompt = f"""
    You are a data analyst. You will receive a dataset profile in JSON.
    Use ONLY the profile to answer questions. 
    Do not ask for the raw data.

    Dataset profile:
    {tabular_summary_str}

    Question 1: Tell me about the dataset.
    """

    tabular_data_message = [{"role": "user", "content": tabular_data_prompt}]

    completion = client.chat.completions.create(
        model=deployment_name,
        messages=tabular_data_message,
        max_tokens=10000,
        temperature=0
    )

    return completion.choices[0].message.content

In [10]:
print(llm_tabular_summary(tabular_summary_str))

The dataset contains information about loans, mortgages, and related financial attributes. Here's a summary:

1. **Structure**:
   - **Rows**: 5,960
   - **Columns**: 13
   - **Duplicate Rows**: None (0% duplicate row fraction)
   - **Time Column**: None (no time-related data)

2. **Columns**:
   - **Numeric Columns**: Most columns are numeric, such as `LOAN`, `MORTDUE`, `VALUE`, `YOJ`, `DEROG`, `DELINQ`, `CLAGE`, `NINQ`, `CLNO`, and `DEBTINC`.
   - **Categorical Columns**: `REASON` and `JOB` are categorical.
   - **Target Column**: `BAD` is a binary numeric column indicating loan default (0 or 1).

3. **Missingness**:
   - **Overall Missingness**: 43.56% of rows have at least one missing value.
   - **Column Missingness**: `DEBTINC` has the highest missing fraction (21.26%), followed by `DEROG` (11.88%) and `DELINQ` (9.73%).

4. **Key Numeric Relationships**:
   - Strong correlation between `MORTDUE` (mortgage due) and `VALUE` (property value), with Pearson correlation coefficient of 